# ImageNet-1k Baseline Pipeline

This notebook is separated from the CIFAR/HBCC workflow and is dedicated to paper-aligned ImageNet-1k baselines. It supports two paths:

1. Official Context-Cluster entrypoint from `docs/context-cluster/train.py`.
2. This repository training/benchmark loop using `configs/imagenet/*.yaml`.

ImageNet must already be prepared locally. The expected structure is `IMAGENET_ROOT/train/<class_id>/*.JPEG` and `IMAGENET_ROOT/val/<class_id>/*.JPEG`. ImageNet-1k test labels are not public, so final reporting uses the official validation split.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "tools" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "tools" / "train.py").exists(), f"Run this notebook from the repo root or notebooks dir. Got {ROOT}"

COC_PYTHON = Path(r"D:\Anaconda\envs\CoC\python.exe")
PYTHON = COC_PYTHON if COC_PYTHON.exists() else Path(sys.executable)

print("Repo:", ROOT)
print("Python:", PYTHON)


## Switches

Set `IMAGENET_ROOT` first, then enable one or more phases.

In [ ]:
IMAGENET_ROOT = "data/imagenet"

RUN_ENV_CHECKS = True
RUN_REPO_IMAGENET_BASELINES = False
RUN_OFFICIAL_COC_ENTRYPOINT = False
RUN_IMAGENET_BENCHMARKS = False
RUN_SUMMARY = True

BENCHMARK_BATCHES = [1, 16, 64, 128]
BENCHMARK_WARMUP = 30
BENCHMARK_RUNS = 100

# Debug settings for quick smoke benchmark only.
DEBUG_BENCHMARK_BATCHES = [1]
DEBUG_BENCHMARK_WARMUP = 1
DEBUG_BENCHMARK_RUNS = 1


In [ ]:
def _looks_like_tqdm(line: str) -> bool:
    text = line.strip()
    return "%|" in text or text.startswith("val:") or text.startswith("test:") or text.startswith("train ")


def run_py(args: list[str], check: bool = True) -> int:
    cmd = [str(PYTHON), *args]
    print("\n$", " ".join(cmd))
    start = time.perf_counter()
    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    assert process.stdout is not None
    progress_handle = display(Markdown(""), display_id=True)
    last_progress = ""
    for line in process.stdout:
        clean = line.rstrip("\r\n")
        if _looks_like_tqdm(clean):
            last_progress = clean
            progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
        else:
            print(line, end="")
    code = process.wait()
    elapsed = time.perf_counter() - start
    if last_progress:
        progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
    print(f"\n[exit={code}] elapsed={elapsed:.1f}s")
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {' '.join(cmd)}")
    return code


def override_args(overrides: list[str] | None) -> list[str]:
    args: list[str] = []
    for item in overrides or []:
        args.extend(["--override", item])
    return args


def train(config: str, output: str, overrides: list[str] | None = None, extra: list[str] | None = None) -> None:
    args = ["tools/train.py", "--config", config, "--output", output]
    args += override_args(overrides or [])
    args += extra or []
    run_py(args)


def benchmark(config: str, checkpoint: str | None = None, debug: bool = False, profile: bool = True) -> None:
    batches = DEBUG_BENCHMARK_BATCHES if debug else BENCHMARK_BATCHES
    warmup = DEBUG_BENCHMARK_WARMUP if debug else BENCHMARK_WARMUP
    runs = DEBUG_BENCHMARK_RUNS if debug else BENCHMARK_RUNS
    args = [
        "tools/benchmark.py",
        "--config", config,
        "--output", "results/benchmark_imagenet",
        "--batch-sizes", *[str(v) for v in batches],
        "--warmup", str(warmup),
        "--runs", str(runs),
    ]
    if checkpoint and (ROOT / checkpoint).exists():
        args += ["--checkpoint", checkpoint]
    elif checkpoint:
        print(f"Skip checkpoint for {config}; not found: {checkpoint}")
    if profile:
        args.append("--profile")
    run_py(args)


## Environment Check

In [ ]:
if RUN_ENV_CHECKS:
    run_py(["-c", "import timm; print(timm.__version__)"])
    official_code = ROOT / "docs" / "context-cluster" / "models" / "context_cluster.py"
    if official_code.exists():
        run_py(["-c", "from lightweight_hbcc.config import load_config; from lightweight_hbcc.models import build_model; cfg=load_config('configs/imagenet/official_coc_tiny_imagenet.yaml'); m=build_model(cfg); print('official_coc_tiny_imagenet', getattr(m, 'num_classes', None), sum(p.numel() for p in m.parameters()))"])
    else:
        print("Skip official CoC check; missing:", official_code)
    for cfg_path in ["configs/imagenet/hbcc_accuracy_small_imagenet.yaml", "configs/imagenet/hbcc_accuracy_medium_imagenet.yaml"]:
        run_py(["-c", f"from lightweight_hbcc.config import load_config; from lightweight_hbcc.models import build_model; cfg=load_config('{cfg_path}'); m=build_model(cfg); print('{cfg_path}', getattr(m, 'num_classes', None), sum(p.numel() for p in m.parameters()))"])
    root = ROOT / IMAGENET_ROOT
    print("ImageNet root:", root)
    print("train exists:", (root / "train").exists())
    print("val exists:", (root / "val").exists())


## Repo Trainer Baselines And HBCC

These jobs use this repository training loop with the selected ImageNet configs: ResNet-18 plus the HBCC accuracy candidates converted from the CIFAR-10 recipes.

In [ ]:
IMAGENET_BASELINE_CONFIGS = [
    "configs/imagenet/resnet18_imagenet.yaml",
    "configs/imagenet/hbcc_accuracy_small_imagenet.yaml",
    "configs/imagenet/hbcc_accuracy_medium_imagenet.yaml",
]

if RUN_REPO_IMAGENET_BASELINES:
    if not (ROOT / IMAGENET_ROOT / "train").exists() or not (ROOT / IMAGENET_ROOT / "val").exists():
        raise FileNotFoundError(f"Prepare ImageNet at {ROOT / IMAGENET_ROOT} with train/ and val/ folders")
    for cfg in IMAGENET_BASELINE_CONFIGS:
        train(cfg, "runs_imagenet", overrides=[f"data.root={IMAGENET_ROOT}"])


## Official Context-Cluster Entrypoint

This path calls `docs/context-cluster/train.py` directly, matching the official code path.

In [ ]:
OFFICIAL_COC_RUN = "runs_imagenet_official/official_coc_tiny_imagenet"
OFFICIAL_COC_CHECKPOINT = f"{OFFICIAL_COC_RUN}/model_best.pth.tar"

if RUN_OFFICIAL_COC_ENTRYPOINT:
    if not (ROOT / IMAGENET_ROOT / "train").exists() or not (ROOT / IMAGENET_ROOT / "val").exists():
        raise FileNotFoundError(f"Prepare ImageNet at {ROOT / IMAGENET_ROOT} with train/ and val/ folders")
    if not (ROOT / "docs" / "context-cluster" / "train.py").exists():
        raise FileNotFoundError("Official Context-Cluster code is missing: docs/context-cluster/train.py")
    run_py([
        "docs/context-cluster/train.py",
        "--data_dir", IMAGENET_ROOT,
        "--train-split", "train",
        "--val-split", "val",
        "--model", "coc_tiny",
        "--num-classes", "1000",
        "--batch-size", "128",
        "--validation-batch-size", "128",
        "--lr", "0.001",
        "--drop-path", "0.1",
        "--amp",
        "--output", "runs_imagenet_official",
        "--experiment", "official_coc_tiny_imagenet",
    ])
    if (ROOT / OFFICIAL_COC_CHECKPOINT).exists():
        run_py([
            "docs/context-cluster/validate.py", IMAGENET_ROOT,
            "--split", "val",
            "--model", "coc_tiny",
            "--num-classes", "1000",
            "--batch-size", "128",
            "--checkpoint", OFFICIAL_COC_CHECKPOINT,
            "--amp",
        ])


## Benchmark

In [ ]:
IMAGENET_BENCHMARK_JOBS = [
    ("configs/imagenet/resnet18_imagenet.yaml", "runs_imagenet/resnet18_imagenet/best.pth"),
    ("configs/imagenet/hbcc_accuracy_small_imagenet.yaml", "runs_imagenet/hbcc_accuracy_small_imagenet/best.pth"),
    ("configs/imagenet/hbcc_accuracy_medium_imagenet.yaml", "runs_imagenet/hbcc_accuracy_medium_imagenet/best.pth"),
]

if RUN_IMAGENET_BENCHMARKS:
    for cfg, ckpt in IMAGENET_BENCHMARK_JOBS:
        benchmark(cfg, checkpoint=ckpt, debug=False, profile=True)


## Summary

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def collect_training_summary(root_pattern: str = "runs_imagenet*/**/metrics.jsonl") -> pd.DataFrame:
    rows = []
    for metrics_path in ROOT.glob(root_pattern):
        records = read_jsonl(metrics_path)
        val_records = [r for r in records if "val_acc1" in r]
        test_records = [r for r in records if "test_acc1" in r]
        if not val_records:
            continue
        best = max(val_records, key=lambda r: r["val_acc1"])
        test = test_records[-1] if test_records else {}
        rows.append({
            "run_dir": str(metrics_path.parent.relative_to(ROOT)),
            "experiment": metrics_path.parent.name,
            "best_epoch": best.get("epoch"),
            "best_val_acc1": best.get("val_acc1"),
            "best_val_acc5": best.get("val_acc5"),
            "test_acc1": test.get("test_acc1"),
            "test_acc5": test.get("test_acc5"),
        })
    return pd.DataFrame(rows).sort_values("best_val_acc1", ascending=False) if rows else pd.DataFrame()


summary_df = collect_training_summary() if RUN_SUMMARY else pd.DataFrame()
summary_df
